<a href="https://www.kaggle.com/code/morescope/speciesnet-validation?scriptVersionId=247151975" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Introduction
As of 2025-06-12, the volunteers at rangers.urbanrivers have added 59,351 observations.  
These observations are not error-proof, but they are definitely a useful means of creating base-truth casses for image labeling.  

We have run speciesnet on all 200k+ images in the database and condensed the results to MongoDB s aiResults. 
This notebook serves to determine the rates of detector confidence and human labels for false negative and false positive results


In [ ]:
# Data Handling
import pandas as pd
from collections import defaultdict

# IO - getting files and images
from pymongo import MongoClient
from kaggle_secrets import UserSecretsClient
import requests
import json
import os
import urllib.parse

# For randomizing which images get downloaded
import random
from tqdm.auto import tqdm

print("==== Loaded Libraries ====")

# Accessing observations (image labels) through pyMongo
This version uses pymongo (MongoClient) 

In [ ]:
%%time
# Get the stored mongo uri secret
user_secrets = UserSecretsClient()
mongo_uri = user_secrets.get_secret("MONGO_PROD")

# Access the server
client = MongoClient(mongo_uri)
db = client['test']
collection = db['cameratrapmedias']

# Fetch documents with at least one speciesConsensus entry
def fetch_all_obs():
    query = {"speciesConsensus.0": {"$exists": True}, "aiResults.0": {"$exists": True}}  # at least one item
    projection = {
        "_id": 0,
        "mediaID": 1,
        "publicURL": 1,
        "speciesConsensus": 1,
        "aiResults": 1
    }
    all_obs = list(collection.find(query, projection))
    print(f"Retrieved {len(all_obs)} documents with speciesConsensus and aiResults.")
    return all_obs

# Try the fetch operation
try:
    print("===== Starting MongoDB Fetch =====")
    obs_json = fetch_all_obs()
except Exception as e:
    print(f"Error during fetch: {e}")

In [ ]:
obs_json[:2]

In [ ]:
# See what observation Types we have
observation_types = set()

for item in obs_json:
    for obs in item.get('speciesConsensus', []):
        obs_type = obs.get('observationType')
        if obs_type is not None:
            observation_types.add(obs_type)

print(observation_types)

In [ ]:
# Flatten to dataframe
flat_rows = []

for item in obs_json:
    media_id = item.get('mediaID')
    url = item.get('publicURL')
    
    # Use first entry of speciesConsensus and aiResults
    consensus = item.get('speciesConsensus', [{}])[0]
    ai = item.get('aiResults', [{}])[0]
    
    flat_rows.append({
        'mediaID': media_id,
        'publicURL': url,
        'observationType': consensus.get('observationType'),
        'scientificName': consensus.get('scientificName'),
        'observationCount': consensus.get('observationCount'),
        'confBlank': ai.get('confBlank'),
        'confHuman': ai.get('confHuman'),
        'confAnimal': ai.get('confAnimal'),
    })

# Create a flat DataFrame
df = pd.DataFrame(flat_rows)
display(df.head())

# Save to CSV file
df.to_csv("observations_and_airesults.csv", index=False)
print("file saved to .csv")

In [ ]:
# How many obs counts are there.
df['observationCount'].value_counts()

## Process the returned JSON for the fields we need
We're looking for the `mediaID` (our primary key),  

What the species consensus from human labeling is,  

And what the aiResult Probabilities are.

In [ ]:
%%time
from collections import defaultdict
import pandas as pd

# Your JSON data here (replace this with your actual data loading step)
data = obs_json  # Replace with your JSON list

# Prepare list for rows
rows = []

for item in data:
    consensus_entries = item.get('speciesConsensus', [])
    ai_result = item.get('aiResults', [{}])[0]  # Assume only one aiResult per mediaID
    
    if not consensus_entries or not ai_result:
        continue

    obs_type = consensus_entries[0].get('observationType')
    
    if obs_type == 'blank':
        group = 'blank'
    elif obs_type == 'human':
        group = 'human'
    elif obs_type == 'animal':
        group = 'animal'
    
    rows.append({
        'group': group,
        'confBlank': ai_result.get('confBlank', 0),
        'confHuman': ai_result.get('confHuman', 0),
        'confAnimal': ai_result.get('confAnimal', 0)
    })

# Convert to DataFrame
df = pd.DataFrame(rows)

# Show the distribution by group
distribution = df.groupby('group').agg(['mean', 'min', 'max'])
with pd.option_context('display.width', 0, 'display.max_colwidth', None):
    display(distribution)


In [ ]:
# Grab results where aiResults > 0.75 vs total
MIN_CONF_ANIMAL = 0.75
MAX_CONF_ANIMAL = 1.00

def fetch_all_obs():
    query = {"aiResults.0.confAnimal": {"$gte": MIN_CONF_ANIMAL, "$lte": MAX_CONF_ANIMAL}}  # Between
    projection = {
        "_id": 0,
        "mediaID": 1,
        "publicURL": 1,
        "speciesConsensus": 1,
        "aiResults": 1
    }
    all_obs = list(collection.find(query, projection))
    print(f"Retrieved {len(all_obs)} documents with aiResults.confAnimal > {MIN_CONF_ANIMAL} and < {MAX_CONF_ANIMAL}.")
    return all_obs

# Try the fetch operation
try:
    print("===== Starting MongoDB Fetch =====")
    filtered_obs_json = fetch_all_obs()
except Exception as e:
    print(f"Error during fetch: {e}")

In [ ]:
filtered_obs_json[:2]

In [ ]:
# Get the max observations count for any media id
filtered_df = pd.DataFrame(filtered_obs_json)

filtered_df['maxObservationCount'] = filtered_df['speciesConsensus'].apply(
    lambda obs_list: max((item.get('observationCount', 0) for item in obs_list), default=0)
    if isinstance(obs_list, list) else 0
)

with pd.option_context('display.width', 0, 'display.max_colwidth', None):
    display(filtered_df.head(1))

In [ ]:
# Get count and percent of items with 
sc_count = len(filtered_df['speciesConsensus'].dropna())
total_count = len(filtered_df)

print(f'Total where confAnimal between {MIN_CONF_ANIMAL} to {MAX_CONF_ANIMAL}: \n{total_count} results')

print(f'\nN results where confAnimal between {MIN_CONF_ANIMAL} to {MAX_CONF_ANIMAL} where at least one specesConsensus exists: \n{sc_count} labels')

## Current Bracket Labeling Completion Rate:

In [ ]:
print("Where at least one speciesConsensus exists...")
print(f'{round(sc_count/total_count*100,2)}% of images are labeled with {MIN_CONF_ANIMAL} < confAnimal < {MAX_CONF_ANIMAL}')

In [ ]:
# Count where at least 3 people have labeled *something* consistently
## This could still be errors or incomplete labels
sc_count_min3 = (filtered_df['maxObservationCount'] >= 3).sum()

print("Where at least one speciesConsensus exists with at least 3 observations ...")
print(f'{round(sc_count_min3/total_count*100,2)}% of images are labeled with {MIN_CONF_ANIMAL} < confAnimal < {MAX_CONF_ANIMAL}')